# Advanced Cross-Validation Methods

This notebook demonstrates advanced CV methods for complex medical datasets:
- Combinatorial Purged Cross-Validation (CPCV)
- Nested Cross-Validation with hyperparameter tuning
- Advanced bootstrap methods
- Multi-level validation strategies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
import sys
sys.path.append('..')
from trustcv.splitters.temporal import CombinatorialPurgedCV
from trustcv.splitters.iid import NestedCV, BootstrapValidation

np.random.seed(42)
plt.style.use('seaborn-v0_8')

## 1. Combinatorial Purged Cross-Validation

CPCV addresses overlapping time periods in financial and clinical data.
It creates multiple combinations of purged training sets.

In [ ]:
# Generate synthetic temporal clinical data
n_samples = 1000
n_features = 20

# Create time-based features
dates = pd.date_range('2020-01-01', periods=n_samples, freq='D')
X = np.random.randn(n_samples, n_features)

# Add temporal dependencies
for i in range(1, n_samples):
    X[i, :5] += 0.3 * X[i-1, :5]  # Add autocorrelation

y = (X[:, 0] + X[:, 1] + 0.5 * np.random.randn(n_samples) > 0).astype(int)

print(f"Data shape: {X.shape}")
print(f"Class distribution: {np.bincount(y)}")
print(f"Temporal span: {dates[0]} to {dates[-1]}")

In [ ]:
# Demonstrate CPCV
cpcv = CombinatorialPurgedCV(n_splits=5, n_test_splits=2, purge_length=7)
model = RandomForestClassifier(n_estimators=50, random_state=42)

scores = []
for fold, (train_idx, test_idx) in enumerate(cpcv.split(X, y)):
    model.fit(X[train_idx], y[train_idx])
    score = model.score(X[test_idx], y[test_idx])
    scores.append(score)
    print(f"Fold {fold + 1}: Score = {score:.3f}")

print(f"\nCPCV Mean Score: {np.mean(scores):.3f} (+/- {np.std(scores) * 2:.3f})")

## 2. Advanced Nested Cross-Validation

Nested CV provides unbiased estimates when doing hyperparameter tuning.

In [ ]:
# Nested CV with hyperparameter tuning
nested_cv = NestedCV(outer_cv=5, inner_cv=3)

param_grid = {
    'n_estimators': [10, 50, 100],
    'max_depth': [3, 5, 7, None]
}

base_model = RandomForestClassifier(random_state=42)
nested_scores = []
best_params_list = []

for fold, (train_idx, test_idx) in enumerate(nested_cv.split(X, y)):
    # Inner loop: hyperparameter tuning
    inner_cv = NestedCV(outer_cv=3, inner_cv=2)
    grid_search = GridSearchCV(
        base_model, param_grid, cv=3, scoring='accuracy'
    )
    
    grid_search.fit(X[train_idx], y[train_idx])
    best_params_list.append(grid_search.best_params_)
    
    # Outer loop: unbiased evaluation
    best_model = grid_search.best_estimator_
    score = best_model.score(X[test_idx], y[test_idx])
    nested_scores.append(score)
    
    print(f"Outer Fold {fold + 1}:")
    print(f"  Best params: {grid_search.best_params_}")
    print(f"  Test score: {score:.3f}")

print(f"\nNested CV Mean Score: {np.mean(nested_scores):.3f} (+/- {np.std(nested_scores) * 2:.3f})")

## Summary

Advanced CV methods provide more robust validation for complex scenarios:

- **CPCV**: Handles temporal dependencies and overlapping periods
- **Nested CV**: Provides unbiased hyperparameter selection
- **Bootstrap methods**: Useful for small datasets

Choose your method based on your data characteristics and computational budget.